In [ ]:
import numpy as np

data = [["I love computer science", 1],
        ["I hate economics", 0],
        ["I love music", 1],
        ["I hate indoor games", 0],
        ["I like reading", 1],
        ["I hate rot learning", 0]]

# 1. Tokenize
tokens = []
labels = []
for sentence, label in data:
    tokens.append(sentence.lower().split())
    labels.append(label)

print("tokens", tokens)
print("labels", labels)

tokens [['i', 'love', 'computer', 'science'], ['i', 'hate', 'economics'], ['i', 'love', 'music'], ['i', 'hate', 'indoor', 'games'], ['i', 'like', 'reading'], ['i', 'hate', 'rot', 'learning']]
labels [1, 0, 1, 0, 1, 0]


In [ ]:
# 2. Vocabulary
vocab_set = set()
for sentence in tokens:
    for word in sentence:
        vocab_set.add(word)
vocabulary = sorted(vocab_set)

vocabulary

['computer',
 'economics',
 'games',
 'hate',
 'i',
 'indoor',
 'learning',
 'like',
 'love',
 'music',
 'reading',
 'rot',
 'science']

In [ ]:
# 3. Word -> index
word_index = {}
for i, word in enumerate(vocabulary):
    word_index[word] = i

word_index

{'computer': 0,
 'economics': 1,
 'games': 2,
 'hate': 3,
 'i': 4,
 'indoor': 5,
 'learning': 6,
 'like': 7,
 'love': 8,
 'music': 9,
 'reading': 10,
 'rot': 11,
 'science': 12}

In [ ]:
# 4. Sentence matrix: one row per sentence, one column per word
vocab_length = len(vocabulary)
sentence_matrix = np.zeros((len(tokens), vocab_length), dtype=int)
for row, sentence in enumerate(tokens):
    for word in sentence:
        sentence_matrix[row][word_index[word]] = 1

sentence_matrix

array([[1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1],
       [0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0],
       [0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0],
       [0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 1, 0]])

In [ ]:
for row in range(len(tokens)):
    present = []
    for j in range(vocab_length):
        if sentence_matrix[row][j] == 1:
            present.append(vocabulary[j])
    print(f"S{row + 1}:", sentence_matrix[row], present)

S1: [1 0 0 0 1 0 0 0 1 0 0 0 1] ['computer', 'i', 'love', 'science']
S2: [0 1 0 1 1 0 0 0 0 0 0 0 0] ['economics', 'hate', 'i']
S3: [0 0 0 0 1 0 0 0 1 1 0 0 0] ['i', 'love', 'music']
S4: [0 0 1 1 1 1 0 0 0 0 0 0 0] ['games', 'hate', 'i', 'indoor']
S5: [0 0 0 0 1 0 0 1 0 0 1 0 0] ['i', 'like', 'reading']
S6: [0 0 0 1 1 0 1 0 0 0 0 1 0] ['hate', 'i', 'learning', 'rot']


In [ ]:
# 5. Lookup table E: one-hot row for each word
E = np.eye(vocab_length, dtype=int)
E

array([[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]])

In [ ]:
# 6. h for one sentence: add up the one-hot rows of its words (the "lookup")
h_lookup = np.zeros(vocab_length, dtype=int)
for word in tokens[0]:
    h_lookup = h_lookup + E[word_index[word]]

# Same thing in matrix form: h = x E
x = sentence_matrix[0]
h = x @ E

print("sentence:", tokens[0])
print("h_lookup:", h_lookup)
print("x @ E:   ", h)
print(np.array_equal(h_lookup, h))   # True

sentence: ['i', 'love', 'computer', 'science']
h_lookup: [1 0 0 0 1 0 0 0 1 0 0 0 1]
x @ E:    [1 0 0 0 1 0 0 0 1 0 0 0 1]
True


In [ ]:
# 7. Weight matrix W: one row per word (input), one column per output node
np.random.seed(0)
out = 3
W = np.random.randn(vocab_length, out)
W.shape   # (vocab_length, 3)

(13, 3)

In [ ]:
# 8. S = h W for one sentence
S = h @ W
print("S:", S)    # shape (3,)

S: [6.02513538 0.2698464  1.08103292]


In [ ]:
# 9. Same for all sentences at once
H = sentence_matrix @ E    # (6, vocab_length), identical to sentence_matrix
S_all = H @ W              # (6, 3): one row of scores per sentence
S_all

array([[ 6.02513538,  0.2698464 ,  1.08103292],
       [ 3.41252943,  2.13327658,  0.92085886],
       [ 2.8436085 ,  0.20008856,  1.95898052],
       [ 2.45539897,  1.60844045,  1.58975962],
       [ 1.56960375,  1.36427373, -1.18608754],
       [-0.49609254, -0.9362893 , -0.49850411]])

In [ ]:
S_loop = np.zeros(out)
for j in range(out):                  # each output node
    total = 0
    for i in range(vocab_length):     # each input (word)
        total += h[i] * W[i][j]       # h1*w1j + h2*w2j + ...
    S_loop[j] = total

print(S_loop)
print(np.allclose(S_loop, S))   # True

[6.02513538 0.2698464  1.08103292]
True
